# MALLORN TDE Classification - Data Exploration

This notebook explores the training data for the MALLORN TDE Classification Challenge.

In [ ]:
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from data_loader import DataLoader
from feature_engineer import LightcurveFeatureEngineer

# Set plot style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

%matplotlib inline
%load_ext autoreload
%autoreload 2

## 1. Load Data

In [ ]:
loader = DataLoader(data_dir='../data/raw')
train_df = loader.load_training_data()

print(f"Dataset shape: {train_df.shape}")
train_df.head()

## 2. Basic Statistics

In [ ]:
# Check for label column
label_col = None
for col in ['label', 'class', 'target']:
    if col in train_df.columns:
        label_col = col
        break

if label_col:
    print(f"Class distribution:")
    print(train_df[label_col].value_counts())
    print(f"\nClass balance:")
    print(train_df[label_col].value_counts(normalize=True))
else:
    print("No label column found")

In [ ]:
# Summary statistics
train_df.describe()

In [ ]:
# Check for missing values
missing = train_df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)

if len(missing) > 0:
    print("Missing values:")
    print(missing)
else:
    print("No missing values found!")

## 3. Visualizations

In [ ]:
# Plot class distribution if label column exists
if label_col:
    fig, ax = plt.subplots(figsize=(8, 5))
    train_df[label_col].value_counts().plot(kind='bar', ax=ax)
    ax.set_title('Class Distribution', fontsize=14, fontweight='bold')
    ax.set_xlabel('Class (0=Non-TDE, 1=TDE)', fontsize=12)
    ax.set_ylabel('Count', fontsize=12)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
# Plot distribution of numeric features
numeric_cols = train_df.select_dtypes(include=[np.number]).columns.tolist()
if label_col and label_col in numeric_cols:
    numeric_cols.remove(label_col)

# Remove ID columns
numeric_cols = [col for col in numeric_cols if 'id' not in col.lower()]

if len(numeric_cols) > 0:
    n_cols = min(4, len(numeric_cols))
    n_rows = min(3, (len(numeric_cols) + n_cols - 1) // n_cols)
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4*n_rows))
    axes = axes.flatten() if n_rows > 1 else [axes] if n_cols == 1 else axes
    
    for i, col in enumerate(numeric_cols[:n_rows*n_cols]):
        train_df[col].hist(bins=50, ax=axes[i], edgecolor='black', alpha=0.7)
        axes[i].set_title(f'Distribution of {col}', fontsize=10)
        axes[i].set_xlabel(col, fontsize=9)
        axes[i].set_ylabel('Frequency', fontsize=9)
    
    # Hide empty subplots
    for i in range(len(numeric_cols), len(axes)):
        axes[i].set_visible(False)
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Correlation matrix (for first few features)
if len(numeric_cols) > 1:
    n_features = min(15, len(numeric_cols))
    corr_matrix = train_df[numeric_cols[:n_features]].corr()
    
    fig, ax = plt.subplots(figsize=(12, 10))
    sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', 
                center=0, square=True, linewidths=1, ax=ax)
    ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 4. Class Comparison

Compare feature distributions between TDE and Non-TDE classes

In [ ]:
if label_col and len(numeric_cols) > 0:
    # Compare distributions for first few features
    n_features_to_plot = min(6, len(numeric_cols))
    
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    axes = axes.flatten()
    
    for i, col in enumerate(numeric_cols[:n_features_to_plot]):
        for class_val in train_df[label_col].unique():
            data = train_df[train_df[label_col] == class_val][col]
            label = 'TDE' if class_val == 1 else 'Non-TDE'
            axes[i].hist(data, bins=30, alpha=0.6, label=label, edgecolor='black')
        
        axes[i].set_title(f'{col}', fontsize=10)
        axes[i].set_xlabel(col, fontsize=9)
        axes[i].set_ylabel('Frequency', fontsize=9)
        axes[i].legend()
        axes[i].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## 5. Next Steps

After exploring the data:
1. Engineer features if needed (in `feature_engineer.py`)
2. Train models (run `python src/train.py`)
3. Generate predictions (run `python src/predict.py`)